# Data Vault 2.0

## Objectifs

Démonstration des principes de base de **Data Vault 2.0** ainsi que :
- **Hubs** : entités principales du cœur de métier (core business entities). Il contiennent une liste unique de clés business (identifiant client) et métadonnées.
- **Links** : les relations entre entités. Connexions entre les **Hubs**
- **Satelites** : pour stocker les attributs et historique





## Mise en place

In [0]:
catalog_name = "demo_" + spark.sql("SELECT current_user()").collect()[0][0].split("@")[0]

silver_schema = "silver"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE {silver_schema}")

## Création des tables pour Data Vault 2.0

### Les **Hubs**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS H_Customer
(
  customer_hk STRING NOT NULL COMMENT 'MD5(customer_id)',
  customer_id INT NOT NULL,
  load_timestamp TIMESTAMP NOT NULL,
  record_source STRING,
  CONSTRAINT pk_h_customer PRIMARY KEY (customer_hk)
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS H_Order
(
  order_hk STRING NOT NULL COMMENT 'MD5(order_id)',
  order_id INT NOT NULL,
  load_timestamp TIMESTAMP NOT NULL,
  record_source STRING,
  CONSTRAINT pk_h_order PRIMARY KEY (order_hk)
);



### Les **Links**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS L_Customer_Order
(
  customer_order_hk STRING NOT NULL COMMENT 'MD5(customer_kh||order_hk)',
  customer_hk STRING NOT NULL,
  order_hk STRING NOT NULL,
  load_timestamp TIMESTAMP NOT NULL,
  record_source STRING,
  CONSTRAINT pk_l_customer_order PRIMARY KEY (customer_order_hk)
);

### Les **Satellites**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS S_Customer
(
  customer_hk STRING NOT NULL,
  hash_diff STRING NOT NULL COMMENT 'MD5 de toutes les colonnes descriptives (non métadata)',
  name STRING,
  address STRING,
  nation_key INT,
  phone STRING,
  acct_bal DECIMAL(12,2),
  market_segment STRING,
  comment STRING,
  load_timestamp TIMESTAMP NOT NULL,
  record_source STRING,
  CONSTRAINT pk_s_customer PRIMARY KEY (customer_hk, load_timestamp)
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS S_Order
(
  order_hk STRING NOT NULL,
  hash_diff STRING NOT NULL COMMENT 'MD5 de toutes les colonnes descriptives (non métadata)',
  order_status STRING,
  total_price DECIMAL(12,2),
  order_date DATE,
  order_priority STRING,
  clerk STRING,
  ship_priority INT,
  comment STRING,
  load_timestamp TIMESTAMP NOT NULL,
  record_source STRING,
  CONSTRAINT pk_s_order PRIMARY KEY (order_hk, load_timestamp)
);

## Chargement des données
1. Chargement des **Hubs**
2. Chargement des **Links**
3. Chargement des **Satellites**

### Fonctions utilitaires

Les fonctions utilitaires (helper functions) sont des fonctions réutilisables qui facilitent le traitement des données, comme le calcul de clés hash, la gestion des timestamps, ou la création de colonnes dérivées. Elles permettent d'automatiser et de standardiser certaines opérations dans le processus de Data Vault.

In [0]:
from pyspark.sql.functions import md5, concat_ws, col

def generate_customer_hash_key(df):
  return df.withColumn(
    "customer_hk",
    md5(col("customer_id").cast("string"))
  )

def generate_order_hash_key(df):
  return df.withColumn(
    "order_hk",
    md5(col("order_id").cast("string"))
  )

def generate_customer_order_hash_key(df):
  return df.withColumn(
    "customer_order_hk",
    md5(concat_ws("||", col("customer_hk"), col("order_hk")))
  )

def generate_hash_diff(df, columns):
  return df.withColumn(
    "hash_diff",
    md5(concat_ws("||", *[col(c) for c in columns]))
  )

### Chargement des tables (**silver**)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.refined_customer
(
  customer_id INT NOT NULL,
  name STRING,
  address STRING,
  nation_key INT,
  phone STRING,
  acct_bal DECIMAL(12,2),
  market_segment STRING,
  comment STRING
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.refined_orders
(
  order_id INT NOT NULL,
  customer_id INT NOT NULL,
  order_status STRING,
  total_price DECIMAL(12,2),
  order_date DATE,
  order_priority STRING,
  clerk STRING,
  ship_priority INT,
  comment STRING
);

In [0]:
from pyspark.sql.functions import col, to_date, current_timestamp

def etl_refined_customer():
    bronze_customer = spark.table("bronze.tpch_customer")
    refined_customer = bronze_customer.select(
        col("c_custkey").cast("int").alias("customer_id"),
        col("c_name").alias("name"),
        col("c_address").alias("address"),
        col("c_nationkey").cast("int").alias("nation_key"),
        col("c_phone").alias("phone"),
        col("c_acctbal").cast("decimal(12,2)").alias("acct_bal"),
        col("c_mktsegment").alias("market_segment"),
        col("c_comment").alias("comment")
    )
    refined_customer.write.mode("overwrite").saveAsTable("silver.refined_customer")


def etl_refined_orders():
    bronze_orders = spark.table("bronze.tpch_orders")
    refined_orders = bronze_orders.select(
        col("o_orderkey").cast("int").alias("order_id"),
        col("o_custkey").cast("int").alias("customer_id"),
        col("o_orderstatus").alias("order_status"),
        col("o_totalprice").cast("decimal(12,2)").alias("total_price"),
        to_date(col("o_orderdate"), "yyyy-MM-dd").alias("order_date"),
        col("o_orderpriority").alias("order_priority"),
        col("o_clerk").alias("clerk"),
        col("o_shippriority").cast("int").alias("ship_priority"),
        col("o_comment").alias("comment")
    )
    refined_orders.write.mode("overwrite").saveAsTable("silver.refined_orders")

In [0]:
etl_refined_customer()
etl_refined_orders()

### Chargement du **Hub** H_Customer

In [0]:
from pyspark.sql.functions import current_timestamp, lit

silver_customer_df = spark.sql("SELECT * FROM silver.refined_customer")

customer_data_hub = (
    generate_customer_hash_key(silver_customer_df)
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("record_source", lit("TPC-H"))
)

customer_data_hub.createOrReplaceTempView("customer_hub_stage")

spark.sql("""
MERGE INTO H_Customer AS target
USING customer_hub_stage AS source
ON target.customer_hk = source.customer_hk
WHEN NOT MATCHED
  THEN INSERT (customer_hk, customer_id, load_timestamp, record_source)
       VALUES (source.customer_hk, source.customer_id, source.load_timestamp, source.record_source)
""")

In [0]:
%sql
select * from H_Customer limit 10

### Chargement du **Satellite** Customer

In [0]:
customer_sat_columns = ["name", "address", "nation_key", "phone", "acct_bal", "market_segment", "comment"]

customer_sat_data = generate_hash_diff(customer_data_hub, customer_sat_columns)
customer_sat_data.createOrReplaceTempView("customer_sat_stage")

spark.sql(f"""
MERGE INTO S_Customer AS target
USING customer_sat_stage AS source
ON target.customer_hk = source.customer_hk
WHEN NOT MATCHED THEN 
INSERT (customer_hk, hash_diff, {', '.join(customer_sat_columns)}, load_timestamp, record_source)
       VALUES (source.customer_hk, source.hash_diff, {', '.join([f'source.{col}' for col in customer_sat_columns])} ,source.load_timestamp, source.record_source)
""")

### Chargement de H_Order et des **Satellites**

In [0]:
silver_orders_df = spark.sql("SELECT * FROM silver.refined_orders")

order_data_hub = (
    generate_order_hash_key(silver_orders_df)
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("record_source", lit("TPC-H"))
)

order_data_hub.createOrReplaceTempView("order_hub_stage")

spark.sql("""
MERGE INTO H_Order AS target
USING order_hub_stage AS source
ON target.order_hk = source.order_hk
WHEN NOT MATCHED
  THEN INSERT (order_hk, order_id, load_timestamp, record_source)
       VALUES (source.order_hk, source.order_id, source.load_timestamp, source.record_source)
""")

order_sat_columns = ["order_status", "total_price", "order_date", "order_priority", "clerk", "ship_priority", "comment"]

order_sat_data = generate_hash_diff(order_data_hub, order_sat_columns)
order_sat_data.createOrReplaceTempView("order_sat_stage")

spark.sql(f"""
MERGE INTO S_Order AS target
USING order_sat_stage AS source
ON target.order_hk = source.order_hk
WHEN NOT MATCHED THEN 
INSERT (order_hk, hash_diff, {', '.join(order_sat_columns)}, load_timestamp, record_source)
       VALUES (source.order_hk, source.hash_diff, {', '.join([f'source.{col}' for col in order_sat_columns])} ,source.load_timestamp, source.record_source)
""")

### Chargement du **Link** Customer-Order

In [0]:
from pyspark.sql.functions import concat_ws

link_data = (
    silver_orders_df.alias("orders")
    .join(spark.table("H_Customer").alias("hc"), on=[col("orders.customer_id") == col("hc.customer_id")], how="inner")
    .join(spark.table("H_Order").alias("ho"), on=[col("orders.order_id") == col("ho.order_id")], how="inner")
    .select(
        col("hc.customer_hk").alias("customer_hk"),
        col("ho.order_hk").alias("order_hk"),
    )
    .withColumn("customer_order_hk", md5(concat_ws("||", col("customer_hk"), col("order_hk"))))
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("record_source", lit("TPC-H"))
)

link_data.createOrReplaceTempView("link_stage")

spark.sql("""
MERGE INTO L_Customer_Order AS target
USING link_stage AS source
ON target.customer_order_hk = source.customer_order_hk
WHEN NOT MATCHED
  THEN INSERT (customer_order_hk, customer_hk, order_hk, load_timestamp, record_source)
       VALUES (source.customer_order_hk, source.customer_hk, source.order_hk, source.load_timestamp, source.record_source)
""")

In [0]:
%sql
select * from link_stage limit 10

## Créations de vues Business

In [0]:
%sql

CREATE OR REPLACE VIEW gold.BV_Customer_Order AS
SELECT
  hc.customer_id,
  sc.name AS customer_name,
  sc.address AS customer_address,
  ho.order_id,
  so.order_date,
  so.total_price,
  so.order_status
FROM
  H_Customer hc
JOIN
  S_Customer sc ON hc.customer_hk = sc.customer_hk
JOIN
  L_Customer_Order lco ON hc.customer_hk = lco.customer_hk
JOIN
  H_Order ho ON lco.order_hk = ho.order_hk
JOIN
  S_Order so ON ho.order_hk = so.order_hk;

## Exemples

In [0]:
%sql
select * from gold.BV_Customer_Order

In [0]:
%sql
-- Ventes totales par Client (customer)
SELECT
  customer_name,
  SUM(total_price) AS total_sales
FROM
  gold.BV_Customer_Order
GROUP BY
  customer_name
ORDER BY
  total_sales DESC;

## Vérifications

In [0]:
%sql
SELECT 'H_Customer' as table_name, COUNT(*) as record_count FROM H_Customer
UNION ALL
SELECT 'S_Customer' as table_name, COUNT(*) as record_count FROM S_Customer
UNION ALL
SELECT 'H_Order' as table_name, COUNT(*) as record_count FROM H_Order
UNION ALL
SELECT 'S_Order' as table_name, COUNT(*) as record_count FROM S_Order
UNION ALL
SELECT 'L_Customer_Order' as table_name, COUNT(*) as record_count FROM L_Customer_Order;

In [0]:
%sql
-- vérification que une commande (order) à bien un seul client (customer)
SELECT
  COUNT(*) as total_orders,
  SUM(CASE WHEN customer_count = 1 THEN 1 ELSE 0 END) AS orders_with_one_customer,
  SUM(CASE WHEN customer_count != 1 THEN 1 ELSE 0 END) AS orders_with_multiple_customers
FROM (
  SELECT order_hk, COUNT(DISTINCT customer_hk) AS customer_count 
  FROM L_Customer_Order 
  GROUP BY order_hk
);